# Imports

In [ ]:
import json
import math
import os
import sys
import time

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from uavloc.data.annotated import AnnotatedImage
from uavloc.datasets import dataset_ref
from uavloc.features.imagenet import normalise
from uavloc.features.segmentation import build_segmenter
from uavloc.plotting.segmentation import centre_window
from uavloc.segmentation.eval import (HOLDOUT_LOCATION, MEAN_KEY, STRIP_LOCATION, STRIP_ROWS, VAL_SETS,
                                      label_at_stride, load_val_sets, model_predictions, results_table, score_sets)
from uavloc.segmentation.instances import building_table, merge_scores, merged_by_gap, status_by_area
from uavloc.segmentation.latency import segmenter_latency, standard_tile
from uavloc.segmentation.losses import CURRENT_LOSS, StructureLossConfig, structure_loss_terms
from uavloc.segmentation.monitor import crop_metrics, tiled_crops
from uavloc.segmentation.preview import log_segmenter_previews
from uavloc.segmentation.topology import track_scores
from uavloc.track import (REPO_ROOT, experiment_dir, log_history, log_run, resolve_device, set_seed,
                          tensorboard_writer)

sys.path.insert(0, str(REPO_ROOT / "pocs" / "village-seg-baseline"))
import village_seg as vs

# Params

In [ ]:
EXP = "021-building-loss-from-scratch"
NOTE = ""
LOG_RUN = True

ARCHITECTURE = "lraspp-mobilenetv3"   # unet-resnet18 | unet-resnet34 | lraspp-mobilenetv3
LOSS = "current"                      # current (CE + Dice) | building | structure (building + track terms)

GAP_WEIGHT = 3.0
GAP_RADIUS_PX = 10
INSTANCE_WEIGHT = 1.0
INSTANCE_REF_PX = 1046
INSTANCE_MAX_RATIO = 10.0
TVERSKY_WEIGHT = 1.0
TVERSKY_ALPHA = 0.3
TVERSKY_BETA = 0.7
SKELETON_WEIGHT = 1.0
SKELETON_DILATION_PX = 1
TRACK_TVERSKY_WEIGHT = 1.0
TRACK_TVERSKY_ALPHA = 0.3
TRACK_TVERSKY_BETA = 0.7

STEPS = 2500
LR = 1e-3
WEIGHT_DECAY = 1e-4
WARMUP_FRAC = 0.05
SAT_CROP_PX = 512
SAT_BATCH = 16
RARE_CROP_P = 0.3
GAP_PX = 64
RECALIBRATE_BN = True
BN_BATCHES = 60
BN_SEED = 123
SEED = 42
DEVICE = "auto"                       # cuda if present, else mps, else cpu
TRAIN_THREADS = 4
LOG_EVERY_STEPS = 50
CHECKPOINT_EVERY_STEPS = 500           # snapshots kept next to the final model; the final step is the one evaluated
MONITOR_TRAIN_CROPS = 90              # as many as the val crops
MONITOR_SEED = 11
EVAL_BATCH = 8
PREVIEW_CROPS = 4
PREVIEW_SEED = 7
DATASET = "ua-village-annotated"
GAP_BINS_M = [0, 1, 2, 3, 4, 6, 8, 12, 16]
AREA_BINS_M2 = [0, 10, 25, 50, 100, 200, 400]
LATENCY_TILE_DATE = "2024-06-06"
LATENCY_TILE_TOP_LEFT_PX = (1024, 1024)

In [ ]:
ORACLE, MODEL = "stride-8 oracle, bilinear", "model"
MERGE_KEYS = ("separable_merge_rate", "resolution_bound_merge_rate", "separate_share", "merged_share", "missed_share")
LOSSES = ("current", "building", "structure")
TAG = f"{ARCHITECTURE}-{LOSS}-seed{SEED}-steps{STEPS}"

# Data

In [ ]:
set_seed(SEED)
torch.set_num_threads(TRAIN_THREADS)
device = resolve_device(DEVICE)
DERIVED = REPO_ROOT / "data" / "derived"
ARTIFACTS = experiment_dir(EXP) / "artifacts"
device

In [ ]:
sat = vs.load_satellite(DERIVED / DATASET, DERIVED)
vs.ban_holdouts(sat, HOLDOUT_LOCATION, STRIP_LOCATION, STRIP_ROWS, GAP_PX)
sat_train = [s for s in sat if not s.banned.all()]
sampler = vs.SatelliteSampler(sat_train, SAT_CROP_PX, RARE_CROP_P)
[(s.loc.split("-")[0], s.date) for s in sat_train]

In [ ]:
val_sets = load_val_sets(DATASET)
val_labels = {name: [a.label for a in images] for name, images in val_sets.items()}
{name: len(images) for name, images in val_sets.items()}

In [ ]:
def centre_crops(images: list[AnnotatedImage], crop_px: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Normalised `(N, 3, P, P)` crops and `(N, P, P)` labels from the centre of each image."""
    rgb = [centre_window(a.rgb[None], crop_px)[0] for a in images]
    labels = [centre_window(a.label[None], crop_px)[0] for a in images]
    x = torch.from_numpy(np.stack([normalise(r).transpose(2, 0, 1) for r in rgb]))
    return x, torch.from_numpy(np.stack(labels).astype(np.int64))

In [ ]:
def round_robin(i: int) -> AnnotatedImage:
    """The i-th preview image: cycle over the val sets, then over each set's dates."""
    images = val_sets[VAL_SETS[i % len(VAL_SETS)]]
    return images[i // len(VAL_SETS) % len(images)]

In [ ]:
previews = {"train": vs.batch_of(sampler, np.random.default_rng(PREVIEW_SEED), PREVIEW_CROPS),
            "val": centre_crops([round_robin(i) for i in range(PREVIEW_CROPS)], SAT_CROP_PX)}
{split: tuple(x.shape) for split, (x, _) in previews.items()}

In [ ]:
monitor = {"train": vs.batch_of(sampler, np.random.default_rng(MONITOR_SEED), MONITOR_TRAIN_CROPS),
           "val": tiled_crops([a for name in VAL_SETS for a in val_sets[name]], SAT_CROP_PX)}
{split: tuple(x.shape) for split, (x, _) in monitor.items()}

# Model

In [ ]:
if LOSS not in LOSSES:
    raise ValueError(f"LOSS must be one of {LOSSES}, got {LOSS!r}")
building_terms = dict(gap_weight=GAP_WEIGHT, gap_radius_px=GAP_RADIUS_PX, instance_weight=INSTANCE_WEIGHT,
                      instance_ref_px=INSTANCE_REF_PX, instance_max_ratio=INSTANCE_MAX_RATIO,
                      tversky_weight=TVERSKY_WEIGHT, tversky_alpha=TVERSKY_ALPHA, tversky_beta=TVERSKY_BETA)
track_terms = dict(skeleton_weight=SKELETON_WEIGHT, skeleton_dilation_px=SKELETON_DILATION_PX,
                   track_tversky_weight=TRACK_TVERSKY_WEIGHT, track_tversky_alpha=TRACK_TVERSKY_ALPHA,
                   track_tversky_beta=TRACK_TVERSKY_BETA)
loss_config = {"current": CURRENT_LOSS, "building": StructureLossConfig(**building_terms),
               "structure": StructureLossConfig(**building_terms, **track_terms)}[LOSS]
loss_config

In [ ]:
model = build_segmenter(ARCHITECTURE, pretrained=True).to(device)
n_params = sum(p.numel() for p in model.parameters())
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=max(STEPS, 1), pct_start=WARMUP_FRAC)
n_params

In [ ]:
def loss_terms(logits: torch.Tensor, y: torch.Tensor) -> dict[str, torch.Tensor]:
    """Every term of the structure loss; `loss` is the one this run optimises."""
    return structure_loss_terms(logits, y, loss_config)

In [ ]:
def training_step(x: torch.Tensor, y: torch.Tensor) -> dict[str, float]:
    """One optimiser step on `loss_terms(...)["loss"]`; returns every term as a float."""
    terms = loss_terms(model(x.to(device)), y.to(device))
    opt.zero_grad()
    terms["loss"].backward()
    opt.step()
    sched.step()
    return {k: v.item() for k, v in terms.items()}

In [ ]:
def monitor_entry(step: int) -> dict[str, float]:
    """Every loss term and metric on the fixed train and val monitor crops, in eval mode."""
    return {"step": step, **{k: v for split, crops in monitor.items()
                             for k, v in crop_metrics(model, crops, loss_terms, EVAL_BATCH, device, split).items()}}

# Run

In [ ]:
writer = tensorboard_writer(EXP)
rng = np.random.default_rng(SEED)
model.train()
history, running, failure, t0 = [monitor_entry(0)], [], None, time.time()
log_history(writer, history[-1])
log_segmenter_previews(writer, model, previews, 0, device)
progress = tqdm(range(1, STEPS + 1), desc=TAG)
try:
    for step in progress:
        lr = sched.get_last_lr()[0]
        terms = training_step(*vs.batch_of(sampler, rng, SAT_BATCH))
        if not math.isfinite(terms["loss"]):
            raise FloatingPointError(f"non-finite loss {terms['loss']} at step {step}")
        running.append(terms)
        progress.set_postfix(loss=f"{terms['loss']:.3f}", val_loss=f"{history[-1]['val_loss']:.3f}",
                             val_miou=f"{history[-1]['val_miou']:.3f}")
        if step % LOG_EVERY_STEPS == 0 or step == STEPS:
            means = {f"train_running_{k}": float(np.mean([r[k] for r in running])) for k in running[0]}
            history.append({**monitor_entry(step), **means, "train_lr": lr, "minutes": (time.time() - t0) / 60})
            log_history(writer, history[-1])
            log_segmenter_previews(writer, model, previews, step, device)
            running = []
        if step % CHECKPOINT_EVERY_STEPS == 0 and step < STEPS:
            torch.save(model.state_dict(), ARTIFACTS / f"model-{TAG}-step{step}.pt")
except Exception as error:  # a crashed run is logged as failed with its partial history, never dropped
    failure = f"{type(error).__name__}: {error}"
train_minutes = (time.time() - t0) / 60
failure, f"{train_minutes:.1f} min"

In [ ]:
if RECALIBRATE_BN and STEPS and failure is None:
    model = vs.domain_bn(model, sampler, SAT_BATCH, BN_BATCHES, BN_SEED, torch.device(device))
model.eval()
final_entry = {**monitor_entry(STEPS + 1), "after_bn": int(RECALIBRATE_BN and STEPS > 0)}
history.append(final_entry)
log_history(writer, final_entry)
log_segmenter_previews(writer, model, previews, STEPS, device, "segmentation_final")
{k: round(v, 4) for k, v in final_entry.items() if k.startswith("val_") and "_iou_" not in k}

In [ ]:
checkpoint = ARTIFACTS / f"model-{TAG}.pt" if STEPS else None
if checkpoint:
    torch.save(model.state_dict(), checkpoint)
checkpoint

In [ ]:
preds = model_predictions(model, val_sets, device)
results = score_sets(preds, val_labels)
results_table(results).round(3)

In [ ]:
def building_rows(name: str, label: np.ndarray, pred: np.ndarray, date: str) -> pd.DataFrame:
    """Per labelled building of one val image: area, gap and status under the stride-8 oracle and the model."""
    maps = {ORACLE: label_at_stride(label, "bilinear"), MODEL: pred}
    return building_table(label, maps).assign(set=name, date=date)

In [ ]:
buildings = pd.concat([building_rows(name, a.label, p, a.date) for name, images in val_sets.items()
                       for a, p in zip(images, preds[name])], ignore_index=True)
merges = {name: merge_scores(frame, MODEL, ORACLE) for name, frame in buildings.groupby("set")}
merges[MEAN_KEY] = {k: float(np.mean([merges[n][k] for n in VAL_SETS])) for k in MERGE_KEYS}
pd.DataFrame(merges).T.round(3)

In [ ]:
by_gap = merged_by_gap(buildings, [ORACLE, MODEL], GAP_BINS_M)
by_gap.round(3)

In [ ]:
by_area = status_by_area(buildings, [MODEL], AREA_BINS_M2)
by_area.round(3)

In [ ]:
tracks = {name: track_scores(preds[name], val_labels[name]) for name in VAL_SETS}
tracks[MEAN_KEY] = {k: float(np.mean([tracks[n][k] for n in VAL_SETS])) for k in tracks[VAL_SETS[0]]}
pd.DataFrame(tracks).T.round(3)

In [ ]:
tile_image = next(a for a in val_sets["bushcha"] if a.date == LATENCY_TILE_DATE)
latency = segmenter_latency(model, standard_tile(tile_image.rgb, *LATENCY_TILE_TOP_LEFT_PX))
latency

# Plots

In [ ]:
frame = pd.DataFrame(history)
CURVES = [("loss", "loss (optimised)"), ("ce", "CE"), ("dice", "Dice"), ("tversky_building", "Tversky, building"),
          ("gap_ce", "CE on gap band"), ("building_ce", "CE on buildings"), ("miou", "mIoU"), ("biou", "Boundary IoU"),
          ("building_iou", "building IoU"), ("building_missed_share", "missed buildings"),
          ("building_merged_share", "merged buildings"), ("mdice", "mean Dice"),
          ("skeleton_recall_track", "skeleton recall loss, track"), ("tversky_track", "Tversky, track"),
          ("track_skeleton_recall", "track centreline recall"), ("track_fragments_per_km", "track fragments per km")]

In [ ]:
fig_curves, axes = plt.subplots(4, 4, figsize=(18, 14))
for ax, (key, title) in zip(axes.ravel(), CURVES):
    for split in ("train", "val"):
        ax.plot(frame["step"], frame[f"{split}_{key}"], "-o", ms=2, label=split)
    ax.set_title(title); ax.set_xlabel("step"); ax.grid(alpha=.25); ax.legend(fontsize=8)
fig_curves.suptitle(f"{ARCHITECTURE}, loss {LOSS}, seed {SEED} (last point: after BN recalibration)")
fig_curves.tight_layout()

In [ ]:
fig_buildings, axes = plt.subplots(1, 2, figsize=(13, 4))
centres = [(a + b) / 2 for a, b in zip(GAP_BINS_M, GAP_BINS_M[1:] + [GAP_BINS_M[-1] + 4])]
for key in (ORACLE, MODEL):
    axes[0].plot(centres, by_gap[key].values, "-o", label=key)
axes[0].set_xlabel("gap to nearest labelled building (m)"); axes[0].set_ylabel("share merged")
labels = [str(b) for b in by_area.index]
for column in (f"{MODEL} missed", f"{MODEL} merged"):
    axes[1].plot(labels, by_area[column].values, "-o", label=column)
axes[1].set_xlabel("labelled building area (m²)"); axes[1].set_ylabel("share"); axes[1].tick_params(axis="x", rotation=30)
for ax in axes:
    ax.grid(alpha=.3); ax.legend(fontsize=8)
fig_buildings.suptitle(f"{ARCHITECTURE}, loss {LOSS}, seed {SEED}")
fig_buildings.tight_layout()

# Log

In [ ]:
mean = results[MEAN_KEY]
metrics = {
    "building_missed_share_mean": merges[MEAN_KEY]["missed_share"],
    "separable_merge_rate_mean": merges[MEAN_KEY]["separable_merge_rate"],
    "miou_mean": mean["miou"],
    "building_iou_mean": float(np.mean([results[n]["iou_per_class"]["building"] for n in VAL_SETS])),
    "boundary_iou_mean": mean["biou"], "bf1_1px_mean": mean["bf1_1px"], "bf1_2px_mean": mean["bf1_2px"],
    "mdice_mean": mean["mdice"],
    "building_merged_share_mean": merges[MEAN_KEY]["merged_share"],
    "building_separate_share_mean": merges[MEAN_KEY]["separate_share"],
    "resolution_bound_merge_rate_mean": merges[MEAN_KEY]["resolution_bound_merge_rate"],
    **{f"{k}_mean": v for k, v in tracks[MEAN_KEY].items()},
    "track_iou_mean": float(np.mean([results[n]["iou_per_class"]["track"] for n in VAL_SETS])),
    "model_cpu_ms": latency.median_ms, "n_params": n_params, "train_minutes": train_minutes,
    "failure": failure,
    **{f"building_missed_share_{n}": merges[n]["missed_share"] for n in VAL_SETS},
    **{f"separable_merge_rate_{n}": merges[n]["separable_merge_rate"] for n in VAL_SETS},
    **{f"building_iou_{n}": results[n]["iou_per_class"]["building"] for n in VAL_SETS},
    **{f"miou_{n}": results[n]["miou"] for n in VAL_SETS},
    **{f"{k}_{n}": v for n in VAL_SETS for k, v in tracks[n].items()},
    **{f"{k}_final": v for k, v in final_entry.items() if k.startswith(("train_", "val_"))},
    "merge_scores": {n: merges[n] for n in VAL_SETS},
    "merged_by_gap": {str(k): v for k, v in by_gap.to_dict(orient="index").items()},
    "status_by_area": {str(k): v for k, v in by_area.to_dict(orient="index").items()},
    "iou_per_class": {n: results[n]["iou_per_class"] for n in VAL_SETS},
    "dice_per_class": {n: results[n]["dice_per_class"] for n in VAL_SETS},
    "biou_per_class": {n: results[n]["biou_per_class"] for n in VAL_SETS},
}
{k: metrics[k] for k in list(metrics)[:5]}

In [ ]:
RUN_CONFIG = {"dataset": dataset_ref(DATASET), "architecture": ARCHITECTURE, "loss": LOSS,
              "loss_config": vars(loss_config), "init": "imagenet", "steps": STEPS, "lr": LR,
              "weight_decay": WEIGHT_DECAY, "warmup_frac": WARMUP_FRAC, "crop_px": SAT_CROP_PX, "batch": SAT_BATCH,
              "rare_crop_p": RARE_CROP_P, "gap_px": GAP_PX, "recalibrate_bn": RECALIBRATE_BN,
              "bn_batches": BN_BATCHES, "bn_seed": BN_SEED, "log_every_steps": LOG_EVERY_STEPS,
              "checkpoint_every_steps": CHECKPOINT_EVERY_STEPS,
              "monitor_train_crops": MONITOR_TRAIN_CROPS, "monitor_seed": MONITOR_SEED, "train_threads": TRAIN_THREADS}

In [ ]:
frame.to_csv(ARTIFACTS / f"history-{TAG}.csv", index=False)
(ARTIFACTS / f"results-{TAG}.json").write_text(json.dumps(
    {"config": RUN_CONFIG, "metrics": metrics, "results": results, "history": history}, indent=1, default=str))
run = log_run(exp=EXP, config=RUN_CONFIG, metrics=metrics, note=NOTE, figures=[fig_curves, fig_buildings],
              status="failed" if failure else "ok", seed=SEED, device=device, writer=writer,
              checkpoint=str(checkpoint.relative_to(REPO_ROOT)) if checkpoint else None) if LOG_RUN else {"run": "dry"}
if not LOG_RUN:
    writer.close()
run["run"]